# H6 — Fase 1: keyATM no documento

Run STM de documento após R1: `youtube_doc_20260927_175555`, treinado com
`stopwords_emojis` e validado pelos testes de resultados. O notebook
interrompe a execução se o run configurado faltar ou as palavras
excluídas ainda aparecerem.

In [ ]:
from pathlib import Path
import sys, json, hashlib, re
import numpy as np
import pandas as pd
from gensim.corpora import Dictionary

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / '03-topic-modeling/configs/params.yaml').exists()), None)
assert PROJECT is not None, 'Execute o notebook dentro do repositorio'
TOPIC_ROOT = PROJECT / '03-topic-modeling'
sys.path.insert(0, str(TOPIC_ROOT / 'notebooks'))
from _helpers import load_params, make_run_output_dir
from _guided import (load_seeds, normalize_seeds, apply_seed_exclusion_rule, build_seed_matrix,
                     category_dependence, bootstrap_prevalence_by_video,
                     coefficient_of_variation, seeded_topic_stability,
                     _run_keyatm_r, guided_nmf, escolher_lambda,
                     seed_topic_assignment, topic_concordance, purity_by_group)

params = load_params(str(TOPIC_ROOT / 'configs/params.yaml'))
guided_cfg = params['guided']
seed_path = TOPIC_ROOT / 'configs' / guided_cfg['seeds_file']
seed_sha256 = hashlib.sha256(seed_path.read_bytes()).hexdigest()
seeds = load_seeds(seed_path)

doc_cfg = params['corpora']['youtube_doc']
seeds = normalize_seeds(seeds, extra_stopwords=doc_cfg.get('stopwords_emojis'))
stm_run = guided_cfg['keyatm'].get('stm_run')
assert stm_run, 'R1 pendente: defina guided.keyatm.stm_run no params.yaml'
stm_dir = TOPIC_ROOT / 'data/output/youtube_doc/stm' / stm_run
required = ['stm_input.csv', 'stm_topics_for_eval.csv', 'stm_theta.csv',
            'stm_results.csv', 'stm_metrics.csv']
assert all((stm_dir / name).is_file() for name in required), stm_dir
topics_stm = pd.read_csv(stm_dir / 'stm_topics_for_eval.csv')
keywords_stm = ' '.join(topics_stm['keywords'].fillna('').astype(str)).lower()
noise = [word for word in doc_cfg['stopwords_emojis']
         if re.search(r'\b' + re.escape(word.lower()) + r'\b', keywords_stm)]
assert not noise, f'R1 falhou: stopwords nos topicos do run {stm_run}: {noise}'
print('Run STM de documento validado:', stm_run)

## Preparar o mesmo texto lematizado do STM e aplicar as sementes congeladas

In [ ]:
stm_input = pd.read_csv(stm_dir / 'stm_input.csv')
assert {'text', 'post_id', 'product_category'} <= set(stm_input.columns)
tokens = stm_input['text'].fillna('').str.split().tolist()
dictionary = Dictionary(tokens)
dictionary.filter_extremes(no_below=params['stm']['no_below'],
                           no_above=params['stm']['no_above'])
vocab = set(dictionary.token2id)
tokens = [[word for word in doc if word in vocab] for doc in tokens]
keep = np.array([bool(doc) for doc in tokens])
tokens = [doc for doc, selected in zip(tokens, keep) if selected]
documents = stm_input.loc[keep].reset_index(drop=True)
assert len(tokens) == len(documents)
effective, exclusions = apply_seed_exclusion_rule(
    seeds, dictionary, guided_cfg['min_df_semente']['youtube_doc'],
    guided_cfg['min_sementes_por_aspecto'])
assert effective, 'Nenhum aspecto sobreviveu a regra de exclusao'
print('Run:', stm_run, 'documentos:', len(documents), 'aspectos:', list(effective))

## Rodar covariáveis, sementes e sensibilidades

In [ ]:
run_dir = make_run_output_dir(TOPIC_ROOT / 'data/output/youtube_doc/guided', 'keyatm')
input_csv = run_dir / 'keyatm_input.csv'
pd.DataFrame({'text': [' '.join(doc) for doc in tokens]}).to_csv(input_csv, index=False)
keywords_json = run_dir / 'keywords.json'
keywords_json.write_text(json.dumps(effective, ensure_ascii=False, indent=2), encoding='utf-8')
(run_dir / 'seeds_efetivas.json').write_text(
    json.dumps({'aspectos': effective, **exclusions}, ensure_ascii=False, indent=2), encoding='utf-8')

categories = documents['product_category'].fillna('outras').astype(str).copy()
video_counts = documents.groupby(categories)['post_id'].nunique()
categories = categories.where(categories.map(video_counts) >=
                              guided_cfg['keyatm']['agrupar_categorias_min_videos'], 'outras')
covariates_csv = run_dir / 'covariates.csv'
pd.DataFrame({'product_category': categories}).to_csv(covariates_csv, index=False)

runs = {}
free_counts = [guided_cfg['keyatm']['no_keyword_topics'],
               *guided_cfg['keyatm']['no_keyword_topics_sens']]
for n_free in free_counts:
    for seed in guided_cfg['seed_runs']:
        subdir = run_dir / f'livres_{n_free}_seed_{seed}'
        result = _run_keyatm_r('covariates', input_csv, keywords_json, subdir,
                               {**guided_cfg['keyatm'], 'no_keyword_topics': n_free},
                               seed=seed, covariates_csv=covariates_csv,
                               covariates_formula='~ product_category')
        theta = pd.read_csv(subdir / result['theta_csv'])
        assert len(theta) == len(documents)
        seeded = list(effective)
        summary = category_dependence(theta.to_numpy(), categories.tolist(),
                                      documents['post_id'].astype(str).tolist(),
                                      min_videos=1)
        summary.to_csv(subdir / 'prevalencia_categoria.csv', index=False)
        mass = {'semeada': float(theta[seeded].sum(axis=1).mean()),
                'livre': float(theta.drop(columns=seeded).sum(axis=1).mean())}
        runs[(n_free, seed)] = {'dir': str(subdir), 'mass': mass,
                                'median_cv_seeded': float(summary[summary.topico < len(seeded)].cv.median()),
                                'median_cv_free': float(summary[summary.topico >= len(seeded)].cv.median())}

metadata = {'stm_doc_run': stm_run, 'seed_file_sha256': seed_sha256,
            'seeds_commit': None, 'effective_seeds': effective,
            'exclusions': exclusions, 'runs': {str(key): value for key, value in runs.items()}}
(run_dir / 'metadata.json').write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saida:', run_dir)

## Leitura humana cega, estabilidade e critério pré-registrado

Preencha `no_aspecto` na planilha antes de consultar as medidas de CV ou massa acima. `sim` para cada tópico semeado exige ≥70% e a mediana CV dos semeados deve ser menor que a dos livres.

In [ ]:
main_seed = guided_cfg['seed_runs'][0]
main_free = guided_cfg['keyatm']['no_keyword_topics']
main_dir = Path(runs[(main_free, main_seed)]['dir'])
main_theta = pd.read_csv(main_dir / 'keyatm_theta.csv')
main_words = pd.read_csv(main_dir / 'topics.csv')
raw_documents = pd.read_csv(stm_dir / 'stm_results.csv', usecols=['post_id', 'text'])
raw_by_id = raw_documents.set_index('post_id')['text'].to_dict()
import spacy
nlp_sentences = spacy.load('en_core_web_sm', disable=['ner'])
rows = []
for aspect, words in effective.items():
    non_seed = [re.sub(r'\s+\[.*\]$', '', str(word)) for word in main_words[aspect].dropna()]
    non_seed = [word for word in non_seed if word not in words][:10]
    row = {'aspecto': aspect, 'top_10_nao_sementes': ', '.join(non_seed)}
    for rank, index in enumerate(main_theta[aspect].nlargest(5).index, start=1):
        post_id = documents.loc[index, 'post_id']
        raw_text = str(raw_by_id.get(post_id, ''))
        sentences = list(nlp_sentences(raw_text).sents) if raw_text else []
        ranked = sorted(sentences, key=lambda sentence: sum(
            token.lemma_.lower() in words for token in sentence), reverse=True)
        excerpt = ' '.join(str(sentence) for sentence in ranked[:3])
        row[f'post_id_{rank}'] = post_id
        row[f'trecho_{rank}'] = excerpt
    row['no_aspecto'] = ''
    rows.append(row)
pd.DataFrame(rows).to_csv(run_dir / 'leitura_humana_cego.csv', index=False)
stability = []
for n_free in free_counts:
    word_runs = []
    for seed in guided_cfg['seed_runs']:
        words = pd.read_csv(Path(runs[(n_free, seed)]['dir']) / 'topics.csv')
        word_runs.append({aspect: [re.sub(r'\s+\[.*\]$', '', str(word))
                                   for word in words[aspect].dropna().head(10)]
                          for aspect in effective})
    table = seeded_topic_stability(word_runs)
    table['n_livres'] = n_free
    stability.append(table)
pd.concat(stability).to_csv(run_dir / 'estabilidade_jaccard.csv', index=False)
print('Preencha:', run_dir / 'leitura_humana_cego.csv')
print('Depois registre o julgamento e as medidas em docs/idealizacao-artigo.txt §5.')